# ViTASA Enhanced — Pair Classification (Colab)

Git clone → Install → Train → Download results

**Thứ tự chạy:** Cell 1 → 2 → 3 (smoke test) → 4 (full training) → 5 (kết quả) → 6 (download)

In [ ]:
# ── Cell 1: Clone repo + install ──────────────────────────────────────────────
import os
if not os.path.exists('/content/VITASA_Enhanced'):
    !git clone https://github.com/Hunganh1305/VITASA_Enhanced.git /content/VITASA_Enhanced
else:
    !git -C /content/VITASA_Enhanced pull

%cd /content/VITASA_Enhanced
!pip install -q torch transformers scikit-learn underthesea

import torch
print(f'✅ PyTorch {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "❌ CPU — vào Runtime → Change runtime type → T4 GPU"}')

In [ ]:
# ── Cell 2: Verify data + run unit tests ──────────────────────────────────────
!ls baseline/data/mobile/ baseline/data/restaurant/ baseline/data/hotel/
!python3 -m pytest text_normalization/tests/ imbalanced_learning/tests/ -q 2>&1 | tail -3

In [ ]:
# ── Cell 3: Smoke test (1 epoch, 10% data) — ~3 phút ─────────────────────────
# Nếu dev F1 > 0% → pipeline OK, chạy Cell 4
!python3 train_pair.py --domain mobile --loss focal --model visobert --epochs 1 --subsample 0.1 --batch-size 64
print('\n✅ Smoke test passed — chạy Cell 4')

In [ ]:
# ── Cell 4: Full ablation ─────────────────────────────────────────────────────
# ⭐ ĐỔI DOMAIN MỖI NGÀY:
#   Ngày 1 → DOMAINS = ['mobile']
#   Ngày 2 → DOMAINS = ['restaurant']
#   Ngày 3 → DOMAINS = ['hotel']
DOMAINS = ['mobile']

# ─────────────────────────────────────────────────────────────────────────────
import subprocess
import time

EPOCHS = 10
BATCH_SIZE = 64
CONFIGS = [
    ('C1_baseline',   '--loss ce'),
    ('C2_norm',       '--loss ce --normalize'),
    ('C3_imbalanced', '--loss focal'),
    ('C4_full',       '--loss focal --normalize'),
]

print(f'Plan: {len(CONFIGS)} configs × {len(DOMAINS)} domains × {EPOCHS} epochs')
print('=' * 70)

failed = []
overall_start = time.time()

for domain in DOMAINS:
    for config_name, flags in CONFIGS:
        print(f'\n[{domain}/{config_name}]')
        cmd = f'python3 train_pair.py --domain {domain} {flags} --model visobert --epochs {EPOCHS} --batch-size {BATCH_SIZE}'
        print(f'CMD: {cmd}')
        print('-' * 70)

        t0 = time.time()
        result = subprocess.run(cmd.split(), cwd='/content/VITASA_Enhanced')
        elapsed = time.time() - t0

        if result.returncode != 0:
            failed.append(f'{domain}/{config_name}')
            print(f'❌ FAILED')
        else:
            print(f'✅ Done in {elapsed/60:.1f} min')

total = time.time() - overall_start
print(f'\n{"=" * 70}')
print(f'Done in {total/60:.1f} min. Failed: {len(failed)}/{len(CONFIGS)*len(DOMAINS)}')
if failed:
    print(f'  Failed: {failed}')
print(f'{"=" * 70}')

In [ ]:
# ── Cell 5: Kết quả ───────────────────────────────────────────────────────────
import json
from pathlib import Path

results_dir = Path('/content/VITASA_Enhanced/experiments/results_pair')
BASELINE = {'mobile': 61.77, 'restaurant': 41.12, 'hotel': 52.64}

rows = []
for f in sorted(results_dir.glob('*/results.json')):
    try:
        d = json.loads(f.read_text())
        rows.append(d)
    except Exception:
        pass

if not rows:
    print('⚠️  Chưa có kết quả — chạy Cell 4 trước')
else:
    print(f'{"-" * 80}')
    print(f'{"Config":<35} {"Domain":<12} {"Dev F1":>8} {"Test F1":>8} {"vs ViTASD":>10}')
    print(f'{"-" * 80}')
    for r in rows:
        domain = r.get('domain', '?')
        test_f1 = r.get('test_f1', 0) * 100
        dev_f1 = r.get('best_dev_f1', 0) * 100
        delta = test_f1 - BASELINE.get(domain, 0)
        print(f'{r.get("config","?"):<35} {domain:<12} {dev_f1:>7.2f}% {test_f1:>7.2f}% {"+" if delta>=0 else ""}{delta:>+9.2f}%')
    print(f'{"-" * 80}')
    print(f'Total: {len(rows)} results')

In [ ]:
# ── Cell 6: Download kết quả ──────────────────────────────────────────────────
from google.colab import files
!find /content/VITASA_Enhanced/experiments/results_pair -name 'results.json' | zip /content/pair_results.zip -@
files.download('/content/pair_results.zip')
print('✅ Download started')